# 05 · 模型评估指标与概率校准 Metrics & Probability Calibration

> **能力标签**：SH7（经典机器学习 / Classical ML）

## 目标 Objectives

完成本课后，你应该能够：

1. 计算**混淆矩阵**（confusion matrix）及衍生指标：Precision、Recall、F1。
2. 绘制 **ROC 曲线**并计算 **AUC**（Area Under the Curve）。
3. 理解**校准曲线**（calibration curve / reliability diagram）并使用 `sklearn.calibration`。
4. 实现 `binary_metrics` 返回 auc、accuracy 及四格混淆矩阵。

> 对应能力：**SH7**

## 概念讲解 Concepts

### 混淆矩阵 Confusion Matrix

|  | 预测正 | 预测负 |
|--|--------|--------|
| **实际正** | TP（真正例） | FN（假负例） |
| **实际负** | FP（假正例） | TN（真负例） |

衍生指标：

| 指标 | 公式 |
|------|------|
| **Accuracy** | $(TP+TN)/N$ |
| **Precision** | $TP/(TP+FP)$ |
| **Recall / Sensitivity** | $TP/(TP+FN)$ |
| **F1** | $2 \cdot P \cdot R / (P+R)$ |
| **Specificity** | $TN/(TN+FP)$ |

---

### ROC 曲线与 AUC

ROC（Receiver Operating Characteristic）曲线以**假正率**（FPR）为 x 轴、**真正率/召回率**（TPR）为 y 轴，遍历所有阈值绘制。

$$\text{AUC} = P(\hat{p}_{pos} > \hat{p}_{neg}) \in [0.5, 1.0]$$

- AUC = 0.5：随机猜测；AUC = 1.0：完美分类。
- 不受类别不平衡影响（相比 accuracy）。

---

### 校准曲线 Calibration Curve

好的分类器不只要**排序准确**（高 AUC），还要**概率校准**（predicted probability ≈ true frequency）。

校准曲线将预测概率分桶，绘制"预测均值 vs 真实阳性率"。对角线 = 完美校准。

- **Platt Scaling**：用 sigmoid 对打分进行后处理校准。
- **Isotonic Regression**：非参校准方法。

```python
from sklearn.calibration import calibration_curve
fraction_of_positives, mean_predicted_value = calibration_curve(y_true, y_prob, n_bins=10)
```

## 示例 Worked Example — `binary_metrics`

In [ ]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import tempfile
from pathlib import Path
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (roc_auc_score, confusion_matrix,
                              roc_curve, ConfusionMatrixDisplay)
from sklearn.calibration import calibration_curve

# ── binary_metrics（镜像 w5-metrics 题包）───────────────────────────────────
def binary_metrics(y_true, y_score, threshold: float = 0.5) -> dict:
    """返回 auc、accuracy 及混淆矩阵四格 (tp,fp,tn,fn)。"""
    y_true  = np.asarray(y_true)
    y_score = np.asarray(y_score, dtype=float)
    y_pred  = (y_score >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    total = tp + tn + fp + fn
    return {"auc":      float(roc_auc_score(y_true, y_score)),
            "accuracy": float((tp + tn) / total),
            "tp": int(tp), "fp": int(fp), "tn": int(tn), "fn": int(fn)}

# ── 数据与模型 ────────────────────────────────────────────────────────────
X, y = make_classification(n_samples=500, n_features=15, n_informative=6,
                            random_state=0)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=0)
clf = LogisticRegression(max_iter=500)
clf.fit(Xtr, ytr)
prob = clf.predict_proba(Xte)[:, 1]

# ── binary_metrics 输出 ───────────────────────────────────────────────────
m = binary_metrics(yte, prob)
print("binary_metrics 结果:")
for k, v in m.items():
    print(f"  {k}: {v}")

# ── 可视化：ROC + 校准曲线 ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

# ROC
fpr, tpr, _ = roc_curve(yte, prob)
axes[0].plot(fpr, tpr, "steelblue", lw=2, label=f"LogReg (AUC={m['auc']:.3f})")
axes[0].plot([0, 1], [0, 1], "k--", lw=1, label="随机猜测 (AUC=0.5)")
axes[0].set_xlabel("假正率 (FPR)")
axes[0].set_ylabel("真正率 (TPR)")
axes[0].set_title("ROC 曲线")
axes[0].legend()

# 校准曲线
frac_pos, mean_pred = calibration_curve(yte, prob, n_bins=10)
axes[1].plot(mean_pred, frac_pos, "s-", color="steelblue", label="LogReg")
axes[1].plot([0, 1], [0, 1], "k--", lw=1, label="完美校准")
axes[1].set_xlabel("平均预测概率")
axes[1].set_ylabel("实际阳性比例")
axes[1].set_title("校准曲线（Reliability Diagram）")
axes[1].legend()

fig.tight_layout()
_tmpdir = tempfile.mkdtemp()
_outpath = Path(_tmpdir) / "metrics_demo.png"
fig.savefig(_outpath, dpi=80)
plt.close(fig)
print("图像已保存到:", _outpath)

## 动手 Exercise

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, confusion_matrix

# 练习：用不同阈值观察 Precision/Recall 权衡
def binary_metrics(y_true, y_score, threshold=0.5):
    y_true  = np.asarray(y_true)
    y_score = np.asarray(y_score, dtype=float)
    y_pred  = (y_score >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    total = tp + tn + fp + fn
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return {"threshold": threshold, "accuracy": (tp+tn)/total,
            "precision": precision, "recall": recall, "f1": f1,
            "tp": int(tp), "fp": int(fp), "tn": int(tn), "fn": int(fn)}

X, y = make_classification(n_samples=400, n_features=12, n_informative=5,
                            weights=[0.7, 0.3], random_state=3)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=0)
clf = LogisticRegression(max_iter=500)
clf.fit(Xtr, ytr)
prob = clf.predict_proba(Xte)[:, 1]

print(f"{'threshold':>10}  {'precision':>10}  {'recall':>8}  {'f1':>8}  {'accuracy':>10}")
print("-" * 55)
for thr in [0.3, 0.4, 0.5, 0.6, 0.7]:
    m = binary_metrics(yte, prob, threshold=thr)
    print(f"{m['threshold']:>10.1f}  {m['precision']:>10.4f}  {m['recall']:>8.4f}  "
          f"{m['f1']:>8.4f}  {m['accuracy']:>10.4f}")
print("\n观察：降低阈值提高 Recall 但降低 Precision（Precision-Recall 权衡）。")

## 延伸阅读 Further Reading

1. **sklearn 指标完整文档**：<https://scikit-learn.org/stable/modules/model_evaluation.html>
2. **sklearn 校准文档**：<https://scikit-learn.org/stable/modules/calibration.html>
3. **《The Elements of Statistical Learning》** — 第 7 章（误差估计）
4. **Fawcett (2006)**："An Introduction to ROC Analysis" — Pattern Recognition Letters

## 关联练习 Related Assignments

```bash
python check.py 04-metrics
```

> 实现 `binary_metrics(y_true, y_score, threshold=0.5)` — 返回 auc、accuracy 及混淆矩阵四格 (tp,fp,tn,fn)。

> 能力标签：**SH7** · threshold ≥ 0.7